In [1]:
import scanpy as sc
# import squidpy as sq
# import pandas as pd
from tqdm.notebook import tqdm
import scipy as sp
import numpy as np
import pickle as pkl
import torch
import gc
import sklearn.metrics

In [2]:
from sklearn import metrics

import matplotlib.pyplot as plt
import seaborn as sns

import os

In [6]:
import sys
sys.path.append("../SEDR")

In [7]:
import SEDR
from sklearn.decomposition import PCA  # sklearn PCA is used because PCA in scanpy is not stable.

## SEDR

In [8]:
# gpu
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [10]:
# aris = []
# nmis = []
name = 'slidetags_sedr_spatial_domain'
random_seed = 2023
SEDR.fix_seed(random_seed)
adata = sc.read_h5ad("../../data/slidetags/HumanTonsil_2000.h5ad")
sc.pp.scale(adata)
adata_X = PCA(n_components=50, random_state=42).fit_transform(adata.X)
adata.obsm['X_pca'] = adata_X
graph_dict = SEDR.graph_construction(adata, 8)
sedr_net = SEDR.Sedr(adata.obsm['X_pca'], graph_dict)
using_dec = True
if using_dec:
    sedr_net.train_with_dec()
else:
    sedr_net.train_without_dec()
sedr_feat, _, _, _ = sedr_net.process()
adata.obsm['SEDR'] = sedr_feat

n_clusters = 3
SEDR.mclust_R(adata, n_clusters, use_rep='SEDR', key_added=name)

adata.obs[[name]].to_csv(f"../Steamboat/revised/saved_results/{name}.csv")

# ari = sklearn.metrics.adjusted_rand_score(adata.obs['parcellation_division'], adata.obs[name])
# nmi = sklearn.metrics.normalized_mutual_info_score(adata.obs['parcellation_division'], adata.obs[name])
# aris.append(ari)
# nmis.append(nmi)

# print(i, adata.obs[name].nunique(), ari, nmi)
# aris.append(ari)
# nmis.append(nmi)

    
# df = pd.DataFrame({'ARI': aris, 'NMI': nmis})
# df.to_csv(f"backup/mmbrain/{name}.csv")
# df

  0%|                                                                                           | 0/200 [00:00<?, ?it/s]/mnt/g/Projects/Steamboat_experiments/../SEDR/SEDR/SEDR_model.py:105: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  total_idx = torch.range(0, self.cell_num-1, dtype=torch.float32).to(self.device)
  0%|                                                                                           | 0/200 [00:00<?, ?it/s]/mnt/g/Projects/Steamboat_experiments/../SEDR/SEDR/SEDR_model.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_kl = F.kl_div(out_q.log(), torch.tensor(tmp_p).to(self.device)).to(self.device)
/home/lshh/miniconda3/envs/sedr/lib/python3

fitting ...
  |======================================================================| 100%


In [11]:
adata.obs[[name]]

,slidetags_sedr_spatial_domain
AAACCCAAGCGCCTTG-1,1
AAACCCAAGTGGACGT-1,1
AAACCCACAGAAGTGC-1,1
AAACCCAGTCATTGCA-1,1
AAACCCATCATCGCAA-1,1
...,...
TTTGTTGCAGGGACTA-1,1
TTTGTTGCATTGTAGC-1,1
TTTGTTGGTACCACGC-1,1
TTTGTTGGTCTGTCCT-1,1
